# 06 — Azure OpenAI Function Calling with MCP Tools

This notebook implements the full six-step lifecycle from notebook 01 by hand: convert MCP tools into an OpenAI function schema, let the model choose a tool, execute it, and get a final answer.

> Requires `AZURE_OPENAI_ENDPOINT` in your `.env` (see notebook 02) and the **Cognitive Services User** role on that resource.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.azure_openai_helper import apply_tool_result, chat_with_tools, get_azure_openai_client
from src.config import get_settings
from src.mcp_client import connect

settings = get_settings()
openai_client = get_azure_openai_client(settings)

In [ ]:
# Step 2: list_tools() -> convert to the OpenAI `tools=` schema.
async def get_tools_schema():
    async with connect(read_only=True) as client:
        return await client.tools_as_openai_functions(), client

# We keep a live connection open for the rest of this notebook so we don't
# reconnect on every cell.
from src.mcp_client import AzureMcpClient, build_server_params
mcp_client = AzureMcpClient(build_server_params(read_only=True))
await mcp_client.__aenter__()
tools_schema = await mcp_client.tools_as_openai_functions()
print(f"{len(tools_schema)} tools available to the model")